# 第五阶段补充：训练标记的分词边界复核

这份 notebook 修正原第五阶段的一个实验设计细节：公开训练代码先单独分词 `[Block-Attention]`，再把 token 放在已分词输入前。原实验将标记和 prefix 拼成字符串再一起分词，token 边界发生了变化。

只针对 LYRA 测试 3 种 prefix × 2 种换行 × 2 种机制，共 12 条回答：无标记、原来的整体分词、与公开代码一致的 token 拼接；图共享位置、独立分块连续位置。它不重跑 CPU FP32，也不修改 v5 类。

请启动新 kernel，再依次运行。默认使用你本次记录的模型/仓库路径和物理 GPU 3。**这份文件没有模型运行结果**；本地只验证了 notebook 格式和代码语法。分开拼接忠实于公开代码的这一步，仍不是整个训练数据流程或权重训练历史的复现。

In [1]:
from pathlib import Path
import os, sys
MODEL_PATH = Path('/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT')
REPO_DIR = Path('/home/lyh/hcy_work/GraphKV_refactor_v5')
GPU_ID = '3'
if 'torch' in sys.modules:
    raise RuntimeError('请重启 kernel，再从本格开始，确保 GPU 选择生效。')
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
RUN_CPU_FP32 = False
CPU_THREADS = 8
MAX_NEW_TOKENS = 64
OUTPUT_DIR = Path.cwd() / 'graphkv_stage5_marker_fix_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
assert (MODEL_PATH/'config.json').is_file(), MODEL_PATH
assert (REPO_DIR/'graph_kv_cache.py').is_file(), '请将 REPO_DIR 改为实际模块目录'
sys.path.insert(0, str(REPO_DIR))
import gc, json, hashlib, inspect, time, re
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from graph_kv_cache import GraphKVCache
from graph_kv_adapter import build_graphkv_from_ids, greedy_generate_graphkv, graphkv_next_logits
assert transformers.__version__ == '4.50.0', transformers.__version__
assert Path(inspect.getfile(GraphKVCache)).resolve().parent == REPO_DIR.resolve()
assert torch.cuda.is_available()
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH),local_files_only=True)
def tok(text):
    return tokenizer(text,return_tensors='pt',add_special_tokens=False,truncation=False).input_ids
SOURCE_TEXTS = {
    '4':'[Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.\n',
    '5':'[Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.\n'}
TARGET_TEXTS = {
    '1':'[Document 1] Project ORION uses the material supplied by ALPHA.\n',
    '2':'[Document 2] Project LYRA uses the material supplied by BETA.\n',
    '3':'[Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.\n'}
SOURCES, TARGETS = ['4','5'], ['1','2','3']
EDGES = [('4','1'),('4','3'),('5','2'),('5','3')]
BASE_PREFIX = '<|user|>\nAnswer using the following documents. Give a short factual answer.\n'
NODES = {n:tok(t) for n,t in {**SOURCE_TEXTS,**TARGET_TEXTS}.items()}
L = max(x.shape[1] for x in NODES.values())
QUESTIONS = {
    'ORION':'What material does ORION use? Answer with the material name only.',
    'LYRA':'What material does LYRA use? Answer with the material name only.',
    'NOVA':'Which two materials does NOVA combine? Answer with the material names only.',
    'BETA_fact':'What material does BETA supply? Answer with the material name only.',
    'LYRA_supplier':'Which supplier supplies the material used by LYRA? Answer with the supplier name only.'}
def query_ids(question, native=False):
    return tok('\nQuestion: '+question+('\n\n' if native else '\n')+'<|assistant|>\n')
results = {'settings':{'model_path':str(MODEL_PATH),'repo_dir':str(REPO_DIR),'gpu':GPU_ID,
    'torch':torch.__version__,'transformers':transformers.__version__, 'L':int(L),
    'max_new_tokens':MAX_NEW_TOKENS,'module_sha256':{n:hashlib.sha256((REPO_DIR/n).read_bytes()).hexdigest()
    for n in ['graph_kv_cache.py','graph_kv_adapter.py']},
    'config_sha256':hashlib.sha256((MODEL_PATH/'config.json').read_bytes()).hexdigest(),
    'source_texts':SOURCE_TEXTS,'target_texts':TARGET_TEXTS,'edges':EDGES,'prefix':BASE_PREFIX,
    'questions':QUESTIONS},'subset':[],'format':[],'fp32':[],'edge_mutation':None}
def save():
    p=OUTPUT_DIR/'graphkv_stage5_results.json'; tmp=p.with_suffix('.tmp')
    tmp.write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding='utf-8'); tmp.replace(p)
def build(model,prefix_ids):
    return build_graphkv_from_ids(model=model, source_inputs={n:NODES[n] for n in SOURCES},
        target_inputs={n:NODES[n] for n in TARGETS}, edges=EDGES,prefix_input_ids=prefix_ids,chunk_span=L)
def digest(g):
    h=hashlib.sha256()
    caches=([g.prefix_cache] if g.prefix_cache is not None else [])+[g.nodes[n] for n in SOURCES+TARGETS]
    for cache in caches:
        for name in ['key_cache','value_cache']:
            for x in getattr(cache,name):
                h.update(x.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())
    return h.hexdigest()
def graph_answer(model,g,q,sources=None,targets=None):
    r=greedy_generate_graphkv(model=model,tokenizer=tokenizer,graph_cache=g,query_input_ids=q,
        source_ids=sources,target_ids=targets,max_new_tokens=MAX_NEW_TOKENS,return_details=True)
    return {'answer':r.text,'token_ids':r.token_ids,'eos':r.stopped_on_eos,
        'hit_limit':len(r.token_ids)==MAX_NEW_TOKENS and not r.stopped_on_eos}
print('已导入：',inspect.getfile(GraphKVCache),'节点长度：',{n:x.shape[1] for n,x in NODES.items()})
save()


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


已导入： /home/lyh/hcy_work/GraphKV_refactor_v5/graph_kv_cache.py 节点长度： {'4': 20, '5': 21, '1': 15, '2': 16, '3': 18}


In [2]:
def layout(prefix_ids, q, attention='graph', positions='shared'):
    chunks = [('prefix', prefix_ids)] + [(n, NODES[n]) for n in SOURCES + TARGETS] + [('query', q)]
    ranges = {}
    pos = []
    ids = []
    offset = 0
    P = int(prefix_ids.shape[1])
    for name, x in chunks:
        n = int(x.shape[1])
        ranges[name] = (offset, offset + n)
        start = offset if positions == 'continuous' else 0 if name == 'prefix' else P if name in SOURCES else P + L if name in TARGETS else P + 2 * L
        pos.extend(range(start, start + n))
        offset += n
        ids.append(x.cpu())
    allow = torch.zeros((offset, offset), dtype=torch.bool)
    if attention == 'causal':
        allow = torch.ones_like(allow).tril()
    else:
        for name, (a, b) in ranges.items():
            allow[a:b, a:b] = torch.ones((b - a, b - a), dtype=torch.bool).tril()
            if name == 'query':
                allow[a:b, :a] = True
            elif attention == 'graph' and name in TARGETS:
                for s, t in EDGES:
                    if t == name:
                        c, d = ranges[s]
                        allow[a:b, c:d] = True
    return (torch.cat(ids, -1), torch.tensor([pos]), allow, ranges)

@torch.inference_mode()
def dense_logits(model, prefix_ids, q, attention='graph', positions='shared'):
    ids, pos, allow, _ = layout(prefix_ids, q, attention, positions)
    dev = model.get_input_embeddings().weight.device
    dtype = next(model.parameters()).dtype
    mask = torch.zeros((1, 1, ids.shape[1], ids.shape[1]), dtype=dtype, device=dev)
    mask.masked_fill_(~allow.to(dev)[None, None], torch.finfo(dtype).min)
    out = model(ids.to(dev), position_ids=pos.to(dev), cache_position=torch.arange(ids.shape[1], device=dev), attention_mask=mask, use_cache=False, return_dict=True)
    return out.logits[:, -1].detach().float()

@torch.inference_mode()
def dense_answer(model, prefix_ids, q, attention, positions):
    q = q.cpu()
    tokens = []
    stopped = False
    eos = model.generation_config.eos_token_id
    if eos is None:
        eos = tokenizer.eos_token_id
    eos = set([] if eos is None else [eos] if isinstance(eos, int) else eos)
    for _ in range(MAX_NEW_TOKENS):
        token = int(dense_logits(model, prefix_ids, q, attention, positions).argmax(-1).item())
        tokens.append(token)
        if token in eos:
            stopped = True
            break
        q = torch.cat([q, torch.tensor([[token]], dtype=torch.long)], -1)
    return {'answer': tokenizer.decode(tokens, skip_special_tokens=True), 'token_ids': tokens, 'eos': stopped, 'hit_limit': len(tokens) == MAX_NEW_TOKENS and (not stopped)}


In [3]:
model=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),local_files_only=True,
    torch_dtype=torch.bfloat16,device_map={'':0},low_cpu_mem_usage=True,attn_implementation='eager').eval()
plain=tok(BASE_PREFIX)
marker_ids=tok('[Block-Attention]')
joint=tok('[Block-Attention]'+BASE_PREFIX)
separate=torch.cat([marker_ids,plain],dim=-1)
assert torch.equal(separate[:,marker_ids.shape[1]:],plain)
prefixes={'no_marker':plain,'joint_string_tokenization':joint,'separate_token_concat':separate}
results['settings']['experiment']='marker_token_boundary_repair'
results['settings']['marker_token_ids']=marker_ids[0].tolist()
results['settings']['prefix_constructions']={name:{'ids':p[0].tolist(),'P':int(p.shape[1]),
    'decoded':tokenizer.decode(p[0].tolist(),skip_special_tokens=False)} for name,p in prefixes.items()}
print('标记独立 token:',marker_ids[0].tolist())
print('原拼接与训练式拼接 token 相同:',torch.equal(joint,separate))
print('各 prefix 长度:',{name:int(p.shape[1]) for name,p in prefixes.items()})
save()
for construction,p in prefixes.items():
    g=build(model,p);initial=digest(g)
    for native in [False,True]:
        q=query_ids(QUESTIONS['LYRA'],native=native)
        for condition in ['v5_graph_shared','independent_continuous']:
            row=graph_answer(model,g,q) if condition=='v5_graph_shared' else dense_answer(model,p,q,'independent','continuous')
            row.update(construction=construction,native=native,question='LYRA',condition=condition,
                prefix_token_ids=p[0].tolist(),query_token_ids=q[0].tolist(),P=int(p.shape[1]))
            results['format'].append(row);save()
            print(construction,native,condition,repr(row['answer']))
    assert digest(g)==initial
    del g
del model;gc.collect();torch.cuda.empty_cache()
print('结果保存到:',OUTPUT_DIR/'graphkv_stage5_results.json')


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.34it/s]


标记独立 token: [58, 4818, 12, 70429, 60]
原拼接与训练式拼接 token 相同: False
各 prefix 长度: {'no_marker': 17, 'joint_string_tokenization': 21, 'separate_token_concat': 22}
no_marker False v5_graph_shared 'Copper.'
no_marker False independent_continuous 'According to Document 3, LYRA uses the material supplied by BETA, which is silver.'
no_marker True v5_graph_shared 'Copper.'
no_marker True independent_continuous 'According to Document 3, LYRA uses the material supplied by BETA, which is silver.'
joint_string_tokenization False v5_graph_shared 'Copper.'
joint_string_tokenization False independent_continuous 'According to Document 3, LYRA uses the material supplied by BETA.'
joint_string_tokenization True v5_graph_shared 'Copper.'
joint_string_tokenization True independent_continuous 'According to Document 3, LYRA uses the material supplied by BETA, which is silver.'
separate_token_concat False v5_graph_shared 'Copper.'
separate_token_concat False independent_continuous 'According to Document 3, LYRA 

如果训练式拼接仍答铜，标记这一解释会进一步减弱；若它能改善，先扩大独立关系样本，再判断是否值得保留。

无标记、整体分词、分别分词的 prefix 长度可能不同，因此后续位置也会改变。即使观察到改善，也需将其理解为输入构造整体变化的效果。回答“BETA 的材料”而没给出银，算未完成材料问题；回答银却引用 Document 3，材料对但引用错。